# 04 Earlier Hub adapter, and logits under the two loading paths

| Section | Content |
|---|---|
| [1] | evaluates the adapter that was at the root of `nextmina/qwen2.5-3b-tombench-qlora` before October 2026 and compares it with the released seed-42 CSVs |
| [2] | loads the seed-42 adapter (`tier4_v2_fixed/adapter_seed42`) with `PeftModel` and with unsloth, compares the quantization and dtype settings, and compares the logits at the answer position on the same inputs |
| [3] | summary |

The earlier root adapter (SHA-256 prefix `b21b4175`) came from a May 2026 development run and is
kept in the Hub repository's commit history; the root now holds the adapter that produced the
released files. To repeat [1], pass the corresponding Hub revision to `snapshot_download`.

Run the cells in order: [0-1] installs the pinned package versions; restart the runtime
(Runtime > Restart session) and continue from [0-2]. Results are written to Google Drive, and
cells that find their output already saved skip it, so an interrupted run can be resumed by
running [0-2] again and then the interrupted cell.

`DRIVE_DIR` is the Drive folder used by `notebooks/tombench_slm_qlora_complete_pipeline.ipynb`;
`tier4_v2_fixed/` inside it holds that notebook's outputs (split and manifest files, per-item
CSVs, adapters). The other folders are created by the revision notebooks; see
`notebooks/revision/README.md` for where their files are in `results/`.

## [0] Setup

In [ ]:
# -- [0-1] Install packages (versions pinned to requirements.lock.txt) --
import time, torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Runtime -> Change runtime type -> select a GPU.'); raise SystemExit
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB VRAM)')
t0 = time.time()
get_ipython().system('pip install -q unsloth==2026.8.18 unsloth_zoo==2026.8.12')
get_ipython().system('pip install -q transformers==5.5.0 trl==0.24.0 accelerate==1.14.0 bitsandbytes==0.50.1 peft==0.20.0')
get_ipython().system('pip install -q "datasets==3.6.0" tqdm')
print(f'   {time.time()-t0:.0f}s  ->  restart the runtime, then run from [0-2]')

In [ ]:
# -- [0-2] Mount Drive, set paths, helpers --
from google.colab import drive
drive.mount('/content/drive')

import os, re, json as _json, time, hashlib, platform
import pandas as pd, numpy as np, torch
import transformers, peft, unsloth
from unsloth import FastLanguageModel
from peft import PeftModel
transformers.logging.set_verbosity_error()
assert transformers.__version__ == '5.5.0', f'transformers version not pinned: {transformers.__version__}'

DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
REPRO_DIR   = f'{DRIVE_DIR}/repro_check'
OUT_DIR     = f'{DRIVE_DIR}/hf_logit_check'
os.makedirs(OUT_DIR, exist_ok=True)
BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
LOCAL_42 = f'{RESULTS_DIR}/adapter_seed42'
HF_REPO = 'nextmina/qwen2.5-3b-tombench-qlora'

def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()[:16]

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)
ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')
def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'
def to_text(tokenizer, user_prompt):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_prompt}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
def load_manifest(bm):
    path = f'{RESULTS_DIR}/split_test_full_v2.json' if bm == 'ToMBench' else f'{RESULTS_DIR}/manifest_{bm}.json'
    with open(path, encoding='utf-8') as f:
        return _json.load(f)
MANIFESTS = {bm: load_manifest(bm) for bm in BENCHMARKS}

@torch.no_grad()
def evaluate_margin(eval_records, model, tokenizer, desc):
    from tqdm.auto import tqdm
    FastLanguageModel.for_inference(model)
    letter_ids = {}
    for L in 'ABCDE':
        ids = tokenizer.encode(L, add_special_tokens=False); assert len(ids) == 1
        letter_ids[ids[0]] = L
    lid = torch.tensor(list(letter_ids.keys()))
    rows = []
    for r in tqdm(eval_records, desc=desc):
        inputs = tokenizer(to_text(tokenizer, r['user_prompt']), return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=12, do_sample=False, pad_token_id=tokenizer.eos_token_id,
                             output_scores=True, return_dict_in_generate=True)
        gen_ids = out.sequences[0][inputs['input_ids'].shape[1]:]
        gen = tokenizer.decode(gen_ids, skip_special_tokens=True)
        pred = extract_letter(gen); margin = None; top2 = None
        for step, tok in enumerate(gen_ids.tolist()):
            if tok in letter_ids and step < len(out.scores):
                v, i = torch.sort(out.scores[step][0].float().cpu()[lid], descending=True)
                margin = float(v[0] - v[1]); top2 = letter_ids[lid[i[1]].item()]; break
        rows.append({'item_id': r['item_id'], 'answer': r['answer'], 'pred': pred, 'raw': gen,
                     'correct': int(pred == r['answer']), 'letter_margin': margin, 'runner_up': top2})
    return pd.DataFrame(rows)

meta = {'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda, 'torch': torch.__version__,
        'transformers': transformers.__version__, 'peft': peft.__version__, 'unsloth': unsloth.__version__,
        'python': platform.python_version(), 'started': time.strftime('%Y-%m-%d %H:%M:%S')}
with open(f'{OUT_DIR}/session_meta.json', 'w') as f: _json.dump(meta, f, indent=2)
print(meta)

## [1] Earlier Hub adapter (unsloth loading)

In [ ]:
# -- [1] HF adapter --
from huggingface_hub import snapshot_download
hf_dir = snapshot_download(repo_id=HF_REPO, local_dir='/content/hf_adapter_seed42')
hf_hash = sha16(f'{hf_dir}/adapter_model.safetensors')
print('HF adapter sha256:', hf_hash, '(earlier Hub adapter: b21b4175343cdfb7)')
print('sha256 of the adapter that produced the CSVs:', sha16(f'{LOCAL_42}/adapter_model.safetensors'), '(1de40b6b5c101d83)')

todo = [bm for bm in BENCHMARKS if not os.path.exists(f'{OUT_DIR}/hf_adapter_{bm}.csv')]
if todo:
    m_hf, t_hf = FastLanguageModel.from_pretrained(model_name=hf_dir, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    for bm in todo:
        evaluate_margin(MANIFESTS[bm], m_hf, t_hf, desc=f'HF-{bm}').to_csv(f'{OUT_DIR}/hf_adapter_{bm}.csv', index=False)
    del m_hf, t_hf; torch.cuda.empty_cache()

rows = []
for bm in BENCHMARKS:
    new = pd.read_csv(f'{OUT_DIR}/hf_adapter_{bm}.csv')
    pub = pd.read_csv(f'{RESULTS_DIR}/finetuned_seed42_{bm}.csv')
    base = pd.read_csv(f'{RESULTS_DIR}/baseline_{bm}.csv')
    m = new.merge(pub[['item_id', 'pred', 'correct']], on='item_id', suffixes=('', '_pub'))
    rows.append({'benchmark': bm, 'n': len(m), 'mismatch_vs_published': int((m['pred'] != m['pred_pub']).sum()),
                 'acc_hf': m['correct'].mean() * 100, 'acc_published': m['correct_pub'].mean() * 100,
                 'delta_hf_vs_base': (m['correct'].mean() - base['correct'].mean()) * 100,
                 'delta_published_vs_base': (m['correct_pub'].mean() - base['correct'].mean()) * 100})
hf = pd.DataFrame(rows)
hf.to_csv(f'{OUT_DIR}/hf_adapter_vs_published.csv', index=False)
print(hf.round(2).to_string(index=False))
print(f"\nTotal mismatches: {hf['mismatch_vs_published'].sum()}/{hf['n'].sum()} "
      f"({hf['mismatch_vs_published'].sum()/hf['n'].sum()*100:.2f}%)")

## [2] The two loading paths

Uses the 67 items that flipped in notebook 02 and 40 items that did not. For each item the logits at
the position after `[[` are taken under both paths; the quantization config, the compute dtype of
the 4-bit layers, the LoRA weight dtype and the LoRA layer classes are also compared.

In [ ]:
# -- [2-1] Load with both paths and compare the configuration --
def describe(model, tag):
    info = {'tag': tag, 'model_class': type(model).__name__}
    qc = getattr(getattr(model, 'config', None), 'quantization_config', None)
    if qc is None and hasattr(model, 'base_model'):
        qc = getattr(model.base_model.model.config, 'quantization_config', None)
    info['quantization_config'] = qc.to_dict() if hasattr(qc, 'to_dict') else str(qc)
    compute, lora_dtypes, lora_cls, base_cls = set(), set(), set(), set()
    for name, mod in model.named_modules():
        if hasattr(mod, 'compute_dtype'): compute.add(str(mod.compute_dtype))
        if 'lora_A' in name and hasattr(mod, 'weight'): lora_dtypes.add(str(mod.weight.dtype))
        if name.endswith('q_proj'):
            lora_cls.add(type(mod).__name__)
            if hasattr(mod, 'base_layer'): base_cls.add(type(mod.base_layer).__name__)
    info.update({'bnb_compute_dtype': sorted(compute), 'lora_weight_dtype': sorted(lora_dtypes),
                 'q_proj_class': sorted(lora_cls), 'q_proj_base_class': sorted(base_cls),
                 'model_dtype': str(getattr(model, 'dtype', None))})
    return info

base_p, tok_p = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
m_peft = PeftModel.from_pretrained(base_p, LOCAL_42); FastLanguageModel.for_inference(m_peft)
m_uns, tok_u = FastLanguageModel.from_pretrained(model_name=LOCAL_42, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
FastLanguageModel.for_inference(m_uns)

cfg = [describe(m_peft, 'PeftModel'), describe(m_uns, 'unsloth')]
with open(f'{OUT_DIR}/load_path_config.json', 'w') as f: _json.dump(cfg, f, indent=2, default=str)
for c in cfg:
    print('\n==', c['tag'])
    for k, v in c.items():
        if k != 'tag': print(f'  {k}: {v}')
same = {k: cfg[0][k] == cfg[1][k] for k in cfg[0] if k != 'tag'}
print('\nSame setting under both paths:', same)

In [ ]:
# -- [2-2] Compare logits directly on the same input --
mm = pd.read_csv(f'{REPRO_DIR}/repro_margin_run1_vs_published.csv')   # run1 (PeftModel, notebook 02) vs released CSVs, with a flipped column
flipped = mm[mm['flipped']]
stable = mm[~mm['flipped']].sample(40, random_state=0)
pick = pd.concat([flipped, stable])
by_id = {it['item_id']: it for bm in BENCHMARKS for it in MANIFESTS[bm]}

letter_ids = [tok_u.encode(L, add_special_tokens=False)[0] for L in 'ABCDE']

@torch.no_grad()
def logits_at_answer(model, tok, user_prompt):
    # position where the answer letter is chosen: prompt + '[[', logits at the last position
    text = to_text(tok, user_prompt) + '[['
    ids = tok(text, return_tensors='pt').to(model.device)
    return model(**ids).logits[0, -1].float().cpu()

rows = []
from tqdm.auto import tqdm
for _, r in tqdm(pick.iterrows(), total=len(pick)):
    up = by_id[r['item_id']]['user_prompt']
    lp = logits_at_answer(m_peft, tok_p, up); lu = logits_at_answer(m_uns, tok_u, up)
    d = (lp - lu).abs()
    lpL, luL = lp[letter_ids], lu[letter_ids]
    rows.append({'item_id': r['item_id'], 'benchmark': r['benchmark'], 'flipped': bool(r['flipped']),
                 'max_abs_diff_vocab': float(d.max()), 'mean_abs_diff_vocab': float(d.mean()),
                 'max_abs_diff_letters': float((lpL - luL).abs().max()),
                 'peft_argmax': 'ABCDE'[int(lpL.argmax())], 'unsloth_argmax': 'ABCDE'[int(luL.argmax())],
                 'unsloth_margin': float(torch.sort(luL, descending=True).values[:2].diff().abs())})
lg = pd.DataFrame(rows)
lg.to_csv(f'{OUT_DIR}/logit_diff_two_paths.csv', index=False)
print(lg.groupby('flipped')[['max_abs_diff_vocab', 'mean_abs_diff_vocab', 'max_abs_diff_letters', 'unsloth_margin']]
      .describe().T.round(5).to_string())
print('\n[2-2] flips reproduced by this method (argmax differs between the two paths):',
      int((lg['peft_argmax'] != lg['unsloth_argmax']).sum()), '/', int(lg['flipped'].sum()), '(of the flipped items)')
del m_peft, base_p, m_uns; torch.cuda.empty_cache()

## [3] Summary (no GPU needed)

In [ ]:
# -- [3] Summary --
from google.colab import drive
drive.mount('/content/drive')
import json as _json, pandas as pd
OUT_DIR = '/content/drive/MyDrive/ToMBench_Results/hf_logit_check'
pd.set_option('display.width', 200)
print('[1] HF adapter vs released CSVs'); print(pd.read_csv(f'{OUT_DIR}/hf_adapter_vs_published.csv').round(2).to_string(index=False))
cfg = _json.load(open(f'{OUT_DIR}/load_path_config.json'))
print('\n[2-1] Loading-path configuration'); [print(c) for c in cfg]
lg = pd.read_csv(f'{OUT_DIR}/logit_diff_two_paths.csv')
print('\n[2-2] Logit differences (flipped=True: items that flipped in notebook 02)')
print(lg.groupby('flipped')[['max_abs_diff_vocab', 'mean_abs_diff_vocab', 'max_abs_diff_letters', 'unsloth_margin']].median().round(5))
print(lg.groupby('flipped')[['max_abs_diff_vocab', 'max_abs_diff_letters']].max().round(5))
print('\nEnvironment:', _json.load(open(f'{OUT_DIR}/session_meta.json')))